In [1]:
from datasets import load_dataset

data_files = {
    "train": "../dataset/train.jsonl",
    "validation": "../dataset/val.jsonl"
}

dataset = load_dataset(
    "json",
    data_files=data_files
)

print(dataset)
print(dataset["train"][0])

/mnt/c/Users/Hamshith/Desktop/Projects/Self_Healing_Daemon/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['messages', 'metadata'],
        num_rows: 1342
    })
    validation: Dataset({
        features: ['messages', 'metadata'],
        num_rows: 167
    })
})
{'messages': [{'role': 'system', 'content': 'You are an expert Site Reliability Engineer analyzing a Kubernetes incident. You have deep knowledge of Kubernetes failure modes, container debugging, and cloud-native systems. Always respond with valid JSON only. No markdown, no explanation outside the JSON.'}, {'role': 'user', 'content': 'Analyze this Kubernetes incident and provide a diagnosis.\n\nINCIDENT DETAILS:\n- Pod Name: event-streamer-xzvc6-qoone\n- Namespace: batch\n- Restart Count: 8\n- Detected At: 2026-03-06T14:29:28.000Z\n\nCONTAINER STATUSES:\n[\n  {\n    "name": "event-container",\n    "ready": false,\n    "restart_count": 8,\n    "state": {\n      "waiting": {\n        "reason": "CrashLoopBackOff",\n        "message": "back-off 20s restarting failed container"\n    

In [2]:
from transformers import AutoTokenizer

# Use the smallest Qwen 2.5 instruct model that is still suitable for
# supervised fine-tuning on a 4 GB RTX 2050.
model_name = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = dataset.map(
    format_chat,
    remove_columns=dataset["train"].column_names,
)

print(dataset["train"][0]["text"])


<|im_start|>system
You are an expert Site Reliability Engineer analyzing a Kubernetes incident. You have deep knowledge of Kubernetes failure modes, container debugging, and cloud-native systems. Always respond with valid JSON only. No markdown, no explanation outside the JSON.<|im_end|>
<|im_start|>user
Analyze this Kubernetes incident and provide a diagnosis.

INCIDENT DETAILS:
- Pod Name: event-streamer-xzvc6-qoone
- Namespace: batch
- Restart Count: 8
- Detected At: 2026-03-06T14:29:28.000Z

CONTAINER STATUSES:
[
  {
    "name": "event-container",
    "ready": false,
    "restart_count": 8,
    "state": {
      "waiting": {
        "reason": "CrashLoopBackOff",
        "message": "back-off 20s restarting failed container"
      }
    },
    "last_state": {
      "terminated": {
        "reason": "OOMKilled",
        "exit_code": 137,
        "source": "last_state"
      }
    }
  }
]

RECENT KUBERNETES EVENTS (last 10 min):
[
  {
    "reason": "OOMKilled",
    "message": "Memory cg

In [3]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)


In [4]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qwen-k8s-diagnosis-3b",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    bf16=False,
    fp16=False,
    use_cpu=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    report_to="none",
    dataloader_num_workers=0,
    remove_unused_columns=False,
)


In [5]:
# Redundant SFTTrainer setup removed to prevent double memory allocation.

In [6]:
import gc
import sys

import torch
from peft import LoraConfig
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from trl import SFTConfig, SFTTrainer

# RTX 2050 has 4 GB VRAM; verify that CUDA is available and that enough
# memory remains for 4-bit model loading and training.
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. Enable NVIDIA GPU support in the notebook kernel."
    )

# Clear stale references before loading the model.
for name in [
    "trainer",
    "model",
    "tokenizer",
    "sft_config",
    "peft_config",
    "bnb_config",
]:
    globals().pop(name, None)

sys.last_value = None
sys.last_tracker = None
sys.last_type = None
sys.last_traceback = None

gc.collect()
torch.cuda.empty_cache()

free_vram_bytes, _ = torch.cuda.mem_get_info()
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Free VRAM after cleanup: {free_vram_bytes / 1024**3:.1f} GB")
# Continue with the 3B QLoRA attempt; CUDA will report an OOM if memory is insufficient.

model_name = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa",
)

# Limit training to a small, memory-efficient LoRA adapter.
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

# SFTConfig is optimized for a small 4 GB GPU.
sft_config = SFTConfig(
    output_dir="./qwen-k8s-diagnosis-3b",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    bf16=False,
    fp16=False,
    use_cpu=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    report_to="none",
    max_length=256,
    dataloader_num_workers=0,
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    peft_config=peft_config,
    processing_class=tokenizer,
    formatting_func=lambda example: example["text"],
)

trainer.train()

# Save the final LoRA adapter and tokenizer after training.
final_output_dir = "./qwen-k8s-diagnosis-3b/final"
trainer.save_model(final_output_dir)
tokenizer.save_pretrained(final_output_dir)


GPU: NVIDIA GeForce RTX 2050
Free VRAM after cleanup: 3.2 GB


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 434/434 [00:11<00:00, 38.57it/s]


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,0.540883,0.507736,0.516732,204800.000000,0.876694
84,0.434953,0.451767,0.468353,343552.000000,0.884537


('./qwen-k8s-diagnosis-3b/final/tokenizer_config.json',
 './qwen-k8s-diagnosis-3b/final/chat_template.jinja',
 './qwen-k8s-diagnosis-3b/final/tokenizer.json')